# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Loaded dataset metadata.\nName: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\nPublished: {dataset.metadata.date_published}")

## 2. Data Overview
Review available record sets, fields, their `@id`s and relevant structure.

In [ ]:
# List record sets and their fields, referencing everything by @id
print('RecordSets and their fields:')
record_sets = dataset.metadata.record_sets
if not record_sets:
    print('No record sets found in the metadata.')
else:
    for rs in record_sets:
        print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
        for field in rs.fields:
            print(f"    - Field: {field.name}, @id: {field.id}, type: {getattr(field, 'data_type', 'n/a')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If there is no record set, the dataset may use data files directly or only one file.

# Collect all record set @id values
record_set_ids = [rs.id for rs in getattr(dataset.metadata, 'record_sets', [])]
# If record sets are absent, try dataset.metadata.fields (single-table case)
if not record_set_ids and hasattr(dataset.metadata, 'fields'):
    # For Croissant 1.0, open data via empty key or None
    record_set_ids = [None]
    print('No record set IDs found; using default (single-table) extraction.')

# Map record sets to DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        dataframes[record_set_id or 'main'] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record_set_id={record_set_id}")
    except Exception as ex:
        print(f"Could not load records for record_set_id={record_set_id}: {ex}")

# Display DataFrame columns for one record set
if dataframes:
    first_id = list(dataframes.keys())[0]
    print(f"Columns for record set {first_id}:\n", dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Use field `@id`s from the schema, e.g., `/cr:Field/abundance` or similar, according to the overview above.

In [ ]:
import numpy as np

# Get the working DataFrame and show numeric fields
df_key = next(iter(dataframes.keys()))
df = dataframes[df_key]

numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
print('Numeric fields:', numeric_candidates)

# Attempt to use a typical quantitative protein field; fallback to first numeric field
if 'abundance' in df.columns:
    numeric_field = 'abundance'
elif numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using found numeric field: {numeric_field}")
else:
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else 0
    filtered_df = df[df[numeric_field] > threshold]

    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())
    
    # Normalization
    filtered_df[f'{numeric_field}_normalized'] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())
    
    # Try grouping by a likely categorical field
    possible_groups = [col for col in df.columns if col not in numeric_candidates and df[col].nunique() < 10 and df[col].dtype == object]
    group_field = possible_groups[0] if possible_groups else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df)
else:
    print('No numeric field available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
    
    # If group_field was selected above, boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


In this notebook, we demonstrated loading and exploring the FAIR² Croissant dataset:

- Loaded dataset metadata, record sets, and fields (by `@id`).
- Extracted data into DataFrames for direct analysis.
- Performed numeric field filtering, normalization, grouping, and visualized data distributions.

This approach can be extended to further data quality assessment, feature engineering, and preparation for machine learning workflows using the Croissant schema and the [`mlcroissant`](https://github.com/mlcommons/croissant) Python API.
